<a href="https://colab.research.google.com/github/un1u3/ml-labs/blob/main/fusemachines-2026/phase3/NEU_Surface_Defect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem Statement 
You are a junior ML engineer at
SmartForge Manufacturing. The production line images thousands of steel strips
every day. Manual visual inspection is slow, inconsistent, and misses subtle
defects that cause costly downstream failures. The engineering team needs two
things:
1. a model that can identify which type of defect is present, and 
2. evidence that the model is robust enough to handle real production variability
(different lighting, orientations, and surface conditions). Your job is to
build and harden that model using PyTorch.

### Setup 

In [2]:
# imports
import torch 
import torch.nn as nn 
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np 

In [3]:
tfm = transforms.Compose([
    transforms.Resize((200,200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]
    )
])
train_ds = datasets.ImageFolder('NEU-DET/train/images',transform=tfm)
test_ds = datasets.ImageFolder('NEU-DET/validation/images',transform=tfm)


In [4]:
print("Number of images:", len(train_ds))
print("Classes:", train_ds.classes)
print("Class to index:", train_ds.class_to_idx)

Number of images: 1440
Classes: ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']
Class to index: {'crazing': 0, 'inclusion': 1, 'patches': 2, 'pitted_surface': 3, 'rolled-in_scale': 4, 'scratches': 5}


In [5]:
train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True
)

In [6]:
train_loader

# Part 0 : NN Foundations 

### 1. Implement a simple 2-layer neural network from scratch
using nn.Module no nn.Sequential shortcuts. Define __init__ and forward
explicitly.


Build: 

1. `__init__ `: explicitly defines layers as attribute 
2. `x.view(x.size(0), -1)` inside forward  since images come in as (batch_size, 3, 200, 200) from the DataLoader, and a Linear layer expects a flat 2D input (batch_size, features),flatten every dimension except the batch dimension. 

3. No activation after fc2 the raw logits are returned directly, because nn.CrossEntropyLoss expects raw logits and applies log_softmax internally. Adding a Softmax here yourself would double-apply it and break training.

In [7]:
class Simple2LayerNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size ):
        super().__init__()

        #layer 1 
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()

    def forward(self, X):
        X = X.view(X.size(0), -1) 
        X = self.fc1(X)
        X = self.relu(X)
        X = self.fc2(X)
        return X


In [8]:
model = Simple2LayerNN(3*200*200,128,output_size=len(train_ds.classes))

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [10]:
for epoch in range(20):
    running_loss = 0.0
    model.train()
    for image, label in train_loader:
        output = model(image)
        loss = criterion(output, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/20, Loss: {avg_loss}")

Epoch 1/20, Loss: 13.727460892995198
Epoch 2/20, Loss: 7.057928821775648
Epoch 3/20, Loss: 4.748158187336392
Epoch 4/20, Loss: 3.832046053144667
Epoch 5/20, Loss: 3.048604267173343
Epoch 6/20, Loss: 2.844674434926775
Epoch 7/20, Loss: 2.859123893578847
Epoch 8/20, Loss: 3.58208460410436
Epoch 9/20, Loss: 3.272056845823924
Epoch 10/20, Loss: 2.114302666982015
Epoch 11/20, Loss: 2.991622769832611
Epoch 12/20, Loss: 1.594138636522823
Epoch 13/20, Loss: 1.3738197402821646
Epoch 14/20, Loss: 1.0562343132164744
Epoch 15/20, Loss: 0.8896818644470639
Epoch 16/20, Loss: 1.0063541581233342
Epoch 17/20, Loss: 0.8483855247497558
Epoch 18/20, Loss: 0.9370537022749583
Epoch 19/20, Loss: 1.5870400233401192
Epoch 20/20, Loss: 1.169628749622239


In [11]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 77.92%


### 2. Activation functions: swap ReLU for Sigmoid in your hidden layer and compare convergence over 20 epochs. What do you notice?

In [12]:
class Simple2LayerNNSigmoid(nn.Module):
    def __init__(self, input_size, hidden_size, output_size ):
        super().__init__()

        #layer 1 
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.sigmoid = nn.Sigmoid()


    def forward(self, X):
        X = X.view(X.size(0), -1) 
        X = self.fc1(X)
        X = self.sigmoid(X)
        X = self.fc2(X)
        return X

In [ ]:
model = Simple2LayerNN(3*200*200,128,output_size=len(train_ds.classes))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
for epoch in range(20):
    running_loss = 0.0
    model.train()
    for image, label in train_loader:
        output = model(image)
        loss = criterion(output, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/20, Loss: {avg_loss}")

model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Reflections: after swaping relu with sigmoid over 20 epochs both I see a drop in accuracy from   `49.72%` to `16.67%`, the convergence is slower the training rate loss drecreae at slower rate,This is because the Sigmoid activation function can suffer from the vanishing gradient problem, making it harder for the network to update its weights effectively. In contrast, ReLU allows gradients to flow more easily, resulting in faster convergence and generally better performance for this image classification task.

### 3.Loss function choice: train with nn.CrossEntropyLoss. Why is this preferred over nn.MSELoss for multi-class defect classification?

In [18]:
model = Simple2LayerNN(3*200*200, 128, output_size=len(train_ds.classes))
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
for epoch  in range(20):
    running_loss = 0.0
    model.train()
    for image, label in train_loader:
        output = model(image)
        loss = criterion(output, label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/20, Loss: {avg_loss}")
        

/home/unique/yes/lib/python3.13/site-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 6])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


RuntimeError: The size of tensor a (6) must match the size of tensor b (32) at non-singleton dimension 1